## **Installing Unsloth and dependencies**
### 1. Optimized Unsloth installation for the Kaggle environment

In [ ]:
import os

os.system(
    "pip install --no-deps unsloth unsloth_zoo"
    " git+https://github.com/huggingface/transformers.git"
)
os.system(
    "pip install --no-deps trl<0.9.0 peft accelerate bitsandbytes xformers"
)
os.system("pip install datasets huggingface_hub evaluate")

## **Authentication, Git configuration, and HF login**

In [ ]:
import subprocess
from google.colab import userdata  # If you are on Colab
from huggingface_hub import login, HfApi, ModelCard, ModelCardData
from kaggle_secrets import UserSecretsError, UserSecretsClient
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

##  **Loading secrets from Kaggle (with fallback to Colab/env)**

In [ ]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    user_email = user_secrets.get_secret("USER_EMAIL")
    user_name = user_secrets.get_secret("USER_NAME")
    hf_username = user_secrets.get_secret("HF_USERNAME")
except Exception:
    try:
        hf_token = userdata.get("HF_TOKEN")
        user_email = userdata.get("USER_EMAIL")
        user_name = userdata.get("USER_NAME")
        hf_username = userdata.get("HF_USERNAME")
    except Exception as e:
        print(f"Warning: Failed to load secrets automatically: {e}")

## **Git configuration settings**

In [ ]:
def set_git_config(email, name):
    try:
        subprocess.run(
            ["git", "config", "--global", "user.email", email], check=True
        )
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git configuration set for: {name} <{email}>")
    except subprocess.CalledProcessError as e:
        print(f"Error setting Git configuration: {e}")

In [ ]:
if user_email and user_name:
    set_git_config(user_email, user_name)

## **Logging in to Hugging Face Hub**

In [ ]:
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    raise ValueError("HF_TOKEN was not found in Kaggle Secrets!")

## **Loading Qwen2.5-Coder-3B & LoRA**

In [ ]:
BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"
HUB_REPO_LORA = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-LoRA"
HUB_REPO_FULL = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-Merged"
HUB_REPO_GGUF = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-GGUF"

In [ ]:
max_seq_length = 2048
dtype = None  # Auto-detection (bfloat16 pro T4/P100 fallback na float16)
load_in_4bit = True

In [ ]:
api = HfApi()
repo_exists = False

#### *Check if a previous checkpoint already exists on the HF Hub*

In [ ]:
try:
    api.model_info(repo_id=HUB_REPO_LORA, token=hf_token)
    repo_exists = True
    print(f"--> Checkpoint found on HF Hub: {HUB_REPO_LORA}. Loading to resume training...")
except Exception:
    print(f"--> Checkpoint not found. I'm starting a new training from base model {BASE_MODEL_NAME}...")

### **RUN INDEX & SEED SETUP**

In [ ]:
RUN_INDEX = 1  # ⚙️ Change manually: 1, 2, 3... (or load automatically)

BASE_SEED = 3407
CURRENT_TRAIN_SEED = BASE_SEED + (RUN_INDEX - 1)

print(f"🚀 RUN INDEX: {RUN_INDEX} | Using Random Seed: {CURRENT_TRAIN_SEED}")

#### *Loading either an adapter in progress or a clean base model.*

In [ ]:
if repo_exists:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=HUB_REPO_LORA,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    # Initialize new QLoRA adapter only on the first run
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", 
            "k_proj", 
            "v_proj", 
            "o_proj",
            "gate_proj", 
            "up_proj", 
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=CURRENT_TRAIN_SEED,
    )

### **LOADING AND PREPARING THE NEO4J DATASET (TEXT-TO-CYPHER)**

#### *Loading the official Neo4j dataset from Hugging Face*

In [ ]:
DATASET_NAME = "neo4j/text2cypher-2025v1"
print(f"Loading dataset: {DATASET_NAME}...")
dataset_raw = load_dataset(DATASET_NAME)

#### *If the dataset has a split (train/test), we take 'train'*

In [ ]:
if "train" in dataset_raw:
  train_dataset = dataset_raw["train"]
else:
  train_dataset = dataset_raw

#### *Function to convert dataset structure to Qwen ChatML format*

In [ ]:
def format_cypher_prompts(examples):
    texts = []
    
    # Extraheujeme question, schema a cypher ze struktury datasetu
    questions = examples["question"]
    schemas = examples["schema"]
    cyphers = examples["cypher"]

    for q, s, c in zip(questions, schemas, cyphers):
        # Compilation of the system report, including the database schema
        system_content = (
            "You are an expert in Neo4j databases and the Cypher query language. "
            "Your task is to convert natural language queries into accurate and efficient Cypher queries "
            "based on the provided database schema.\n\n"
            f"### Database Schema:\n{s}"
        )

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": q},
            {"role": "assistant", "content": f"```cypher\n{c}\n```"},
        ]

        # Formátování do Qwen ChatML šablony
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

#### *Applying formatting to the entire dataset*

In [ ]:
print("I am formatting the data into the ChatML structure...")
dataset = train_dataset.map(format_cypher_prompts, batched=True)

In [ ]:
print(f"Dataset prepared! Total training examples: {len(dataset)}")

### **Training configuration (SFTTrainer with epoch definition)**

#### *Target training parameters*

In [ ]:
TARGET_EPOCHS = 3  # Set the total number of epochs required

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=CURRENT_TRAIN_SEED,
        data_seed=CURRENT_TRAIN_SEED,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

#### *Input masking (training only on the response – Cypher code)*

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

### *Starting the training session, with the option to resume*

In [ ]:
trainer_stats = trainer.train()

### **Conditional saving (LoRA vs. Merged 16-bit & GGUF)**

In [ ]:
current_epoch = trainer.state.epoch
print(f"Epochs currently completed: {current_epoch} / {TARGET_EPOCHS}")

#### *Always save/update intermediate LoRA adapters*

In [ ]:
print(f"Saving intermediate LoRA adapters to: {HUB_REPO_LORA}")
model.push_to_hub(HUB_REPO_LORA, token=hf_token)
tokenizer.push_to_hub(HUB_REPO_LORA, token=hf_token)

#### *Check whether the final number of epochs has been reached*

In [ ]:
if current_epoch >= TARGET_EPOCHS:
    print("\n" + "=" * 50)
    print("Target epochs reached! Exporting 16-bit merged and GGUF models...")
    print("=" * 50)

    # 1. Push 16-bit Merged Model
    print(f"1. Uploading merged 16-bit model to: {HUB_REPO_FULL}")
    model.push_to_hub_merged(
        HUB_REPO_FULL,
        tokenizer,
        save_method="merged_16bit",
        token=hf_token,
    )

    # 2. Push GGUF Model (Q4_K_M)
    print(f"2. Converting and uploading GGUF (Q4_K_M) to: {HUB_REPO_GGUF}")
    model.push_to_hub_gguf(
        HUB_REPO_GGUF,
        tokenizer,
        quantization_method="q4_k_m",
        token=hf_token,
    )

    # 3. Create and Upload Model Card
    print("3. Generating and uploading Hugging Face Model Card...")

    # Safely obtaining the number of examples
    dataset_size = len(dataset) if "dataset" in locals() else len(train_dataset)
    
    card_data = ModelCardData(
        language="en",
        license="apache-2.0",
        library_name="peft",
        tags=["text-to-cypher", "neo4j", "cypher", "unsloth", "qwen2.5"],
        datasets=[DATASET_NAME],
        pipeline_tag="text-generation",
    )

    model_card_text = f"""
# Qwen2.5-Coder-3B Text-to-Cypher (Neo4j)

This model is a fine-tuned version of **{BASE_MODEL_NAME}** optimized for translating natural language questions into accurate **Neo4j Cypher queries** using a structured database schema.

## Model Details

- **Base Architecture:** Qwen2.5-Coder-3B-Instruct
- **Fine-Tuning Method:** QLoRA (Rank 16, Alpha 16)
- **Dataset Used:** `{DATASET_NAME}` ({dataset_size} examples)
- **Epochs:** {TARGET_EPOCHS}
- **Frameworks:** Unsloth, Hugging Face TRL, PEFT, PyTorch

## Prompts & Schema Usage

The model expects system prompts containing the Neo4j schema structure:

```text
<|im_start|>system
You are an expert in Neo4j databases and the Cypher query language. Your task is to convert natural language queries into accurate and efficient Cypher queries based on the provided database schema.

### Database Schema:
[PROVIDE DATABASE SCHEMA HERE]<|im_end|>
<|im_start|>user
[YOUR NATURAL LANGUAGE QUESTION]<|im_end|>
<|im_start|>assistant
```

## Running with Ollama (GGUF)
You can run the exported GGUF model ({HUB_REPO_GGUF}) directly in Ollama:

```bash
ollama run hf.co/{HUB_REPO_GGUF}:Q4_K_M
```
"""
    # Push README to both Merged and GGUF repositories
    for repo_id in [HUB_REPO_FULL, HUB_REPO_GGUF, HUB_REPO_LORA]:
        try:
            card = ModelCard(model_card_text, ignore_metadata_errors=True)
            card.data = card_data
            card.push_to_hub(repo_id, token=hf_token)
            print(f"📝 Model Card successfully updated for: {repo_id}")
        except Exception as e:
            print(f"Failed to update Model Card for {repo_id}: {e}")

    print("\n✅ All operations completed successfully!")

else:
    print(
        f"\nTraining is not yet complete ({current_epoch}/{TARGET_EPOCHS} epochs). "
        "Only the intermediate LoRA version was updated."
    )